
Tiny Shakespeare veri setini okumak, karakter düzeyinde tokenizer (encode/decode) geliştirmek, veriyi %90 train - %10 validation olarak ayırmak, rastgele parçalardan batch çeken `get_batch` fonksiyonunu yazmak ve `BigramLanguageModel` sınıfını `nn.Module` olarak kurup eğiterek karşılaştırma tabanımızı (baseline validation loss) elde etmek.

> **Referans:** Andrej Karpathy - "Let's build GPT: from scratch, in code, spelled out" (Video: 7:52 - 38:00)

### 1. Veriyi Okuma ve İnceleme

In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("toplam karakter sayisi:", len(text))
print("ilk 200 kelime:\n",text[:200])


toplam karakter sayisi: 1115394
ilk 200 kelime:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Karakterler:", ''.join(chars))
print(vocab_size)

Karakterler: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


In [6]:
sample_text = "selam yzElli"
encoded = encode(sample_text)
decoded = decode(encoded)

print(sample_text)
print(encoded)
print(decoded)

selam yzElli
[57, 43, 50, 39, 51, 1, 63, 64, 17, 50, 50, 47]
selam yzElli


In [7]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(len(data))
print(len(train_data))
print(len(val_data))

1115394
1003854
111540


### Bağlam Uzunluğu (block_size) 

In [ ]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"{context.tolist()} ('{decode(context.tolist())}') ---> {target.item()} ('{decode([target.item()])}')")

[18] ('F') ---> 47 ('i')
[18, 47] ('Fi') ---> 56 ('r')
[18, 47, 56] ('Fir') ---> 57 ('s')
[18, 47, 56, 57] ('Firs') ---> 58 ('t')
[18, 47, 56, 57, 58] ('First') ---> 1 (' ')
[18, 47, 56, 57, 58, 1] ('First ') ---> 15 ('C')
[18, 47, 56, 57, 58, 1, 15] ('First C') ---> 47 ('i')
[18, 47, 56, 57, 58, 1, 15, 47] ('First Ci') ---> 58 ('t')


### 6. Veriden Rastgele Batch Çeken `get_batch` Fonksiyonu

In [9]:
torch.manual_seed(1337)
batch_size = 32
block_size = 8 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print(xb.shape)
print(yb.shape)

device: cuda
torch.Size([32, 8])
torch.Size([32, 8])


In [10]:
import torch.nn as nn
from torch.nn import functional as F


### 7. BigramLanguageModel (nn.Module)

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
            
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx:(B, T)
        for _ in range(max_new_tokens):
            # tahminleri al
            logits, loss = self(idx)
            # son kelime
            logits = logits[:, -1, :] # (B, C)
            # olasılıklar için softmax 
            probs = F.softmax(logits, dim=-1) # (B, C)
            # bir sonraki tokeni tahminle
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # yeni tokeni ekle
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1), her batch ayrı bir satır
        return idx

### loss Fonksiyonu

In [ ]:
eval_iters = 200

@torch.no_grad()
def estimate_loss(model):
    out = {} # loss values
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

### Bigram Modelinin Eğitilmesi

In [ ]:
torch.manual_seed(42)
model = BigramLanguageModel(vocab_size)
model = model.to(device)

# Hiperparameters
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Eğitim Başlıyor...")
for iter in range(max_iters):

    # Belirli aralıklarla train ve validation kayıplarını değerlendir
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {iter:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Train batch çek
    xb, yb = get_batch('train')

    # Kaybı hesapla ve geriye yayılım uygula
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True) # gradyanı sıfırlamak yerine direkt bellekteki yerini boşaltıyor
    loss.backward()
    optimizer.step()

Eğitim Başlıyor...
step    0: train loss 4.7627, val loss 4.7633
step  300: train loss 2.8415, val loss 2.8635
step  600: train loss 2.5515, val loss 2.5881
step  900: train loss 2.4970, val loss 2.5305
step 1200: train loss 2.4908, val loss 2.5049
step 1500: train loss 2.4716, val loss 2.5028
step 1800: train loss 2.4557, val loss 2.4971
step 2100: train loss 2.4725, val loss 2.4942
step 2400: train loss 2.4637, val loss 2.4956
step 2700: train loss 2.4644, val loss 2.4894
step 2999: train loss 2.4624, val loss 2.4929


### Karşılaştırma  ve Metin Üretimi

In [ ]:
from pandas._libs import interval
# değerlendirme
baseline_losses = estimate_loss(model)

print(f"train_loss: {baseline_losses['train']:.4f}")
print(f"Val_loss: {baseline_losses['val']:.4f}")


# 500 tokelik kelime üretimi
context = torch.zeros((1, 1), dtype=torch.long, device=device) # indeks ararken int64 veya long
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print("\nörnek metin\n")
print(generated_text)

train_loss: 2.4566
Val_loss: 2.4911

örnek metin


Wourr hok isone theeer en, t:
POMPO:

Ast stubad wie,
Tond pand Voree m h d had mys pumade,
amverd,

Whipu blainou s, wine:
Joundis busaveset, arund ticke?
I s
I, s m:
Yo ildo t l ake boo ateerouthyoofr br, o tbe thie; po lkise wir sthay fol-bitatuthat gakesMo mandjoursang h bu'sm, k I t:
AParetin,
T:
Hee joro,
PUK: bousor dom'st I thathe, be o be whers amasesthoul mee'd tshakis t lou s waio,
HAs!
fr are t rerd:

Eves sme, ir frs BUCEO I id s hind, I t.famidely d me nn, thel, Tow, mineallonoded,
